In [1]:
# Parameters
BATCH_MODE = "true"

print(f"BATCH_MODE = {BATCH_MODE}")


BATCH_MODE = true


# SC-14-Formal-Verification - Verification Formelle

[<< Precedent : Fuzz & Invariants](SC-13-Fuzz-Invariants.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Zero-Knowledge Proofs >>](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les principes de la **verification formelle**
2. Utiliser **Certora Prover** et le langage CVL
3. Ecrire des **specifications** et des **regles**
4. Verifier des invariants mathematiques

### Prerequis

- [SC-9](SC-12-Foundry-Testing.ipynb) et [SC-10](SC-13-Fuzz-Invariants.ipynb) completes
- Notions de logique formelle (optionnel)
- Foundry installe, acces a Certora ou Halmos recommande

### Duree estimee : 50 minutes

---

## 1. Introduction a la Verification Formelle

La verification formelle prouve mathematiquement la correction d'un programme.

In [2]:
# Concepts de verification formelle
print("""
VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)
""")


VERIFICATION FORMELLE vs TESTS

| Aspect           | Tests          | Verification Formelle |
|------------------|----------------|----------------------|
| Couverture       | Partielle      | Complete (exhaustive) |
| Method           | Exemples       | Preuves mathematiques |
| Complexite       | Simple         | Elevee               |
| Cout             | Faible         | Eleve                |
| Faux positifs    | Non            | Possible             |

OUTILS DISPONIBLES:
- Certora Prover (commercial, puissant)
- SMTChecker (Solidity builtin)
- KEVM (K framework)
- Act (formal verification for Move)



### Interpretation : Verification formelle vs Tests

**Resultat obtenu** : Comparaison des approches de validation.

| Critere | Tests (unitaires/fuzz) | Verification formelle |
|---------|------------------------|----------------------|
| **Couverture** | Partielle (echantillonnage) | Complete (exhaustive) |
| **Methodologie** | Exemples concrets | Preuves mathematiques |
| **Complexite** | Simple a mettre en place | Requires expertise |
| **Cout** | Faible | Eleve (temps, ressources) |
| **Faux positifs** | Non | Possibles (specs incorrectes) |
| **Faux negatifs** | Possibles (cas non testes) | Non (si spec correcte) |

**Points cles** :
- La verification formelle prouve qu'une propriete est vraie pour TOUTES les entrees possibles
- Les tests ne peuvent que verifier des cas specifiques
- Certora Prover est un outil commercial puissant utilisant le langage CVL
- SMTChecker est une alternative gratuite integree au compilateur Solidity
- Les deux approches sont complementaires : tests pour developpement rapide, verification formelle pour contrats critiques

---

## 2. Certora Prover

Certora utilise le CVL (Certora Verification Language).

In [3]:
# Contrat Solidity a verifier
SOL_CONTRACT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}
'''

print("Contrat Vault a verifier:")
print(SOL_CONTRACT)

Contrat Vault a verifier:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Vault {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;
        payable(msg.sender).transfer(amount);
    }

    function getBalance(address user) external view returns (uint256) {
        return balances[user];
    }
}



Spécification formelle CVL (Certora Verification Language) définissant les propriétés à vérifier sur le contrat Vault.

In [4]:
# Specification CVL pour Vault
CVL_SPEC = '''
// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);
    
    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();
    
    // Verifier le resultat
    assert balances(user) == balanceBefore + amount;
}

// Regle: withdraw reduit correctement la balance
rule withdrawDecreasesBalance(address user, uint256 amount) {
    require amount > 0;
    require balances(user) >= amount;
    
    uint256 balanceBefore = balances(user);
    uint256 totalBefore = totalDeposits();
    
    env.msgSender = user;
    withdraw(amount);
    
    assert balances(user) == balanceBefore - amount;
    assert totalDeposits() == totalBefore - amount;
}
'''

print("Specification CVL:")
print(CVL_SPEC)

Specification CVL:

// Fichier: Vault.spec

// Definition du contrat a verifier
methods {
    function balances(address) external returns (uint256) envfree;
    function totalDeposits() external returns (uint256) envfree;
    function deposit() external payable;
    function withdraw(uint256) external;
}

// Invariant global: sum des balances == totalDeposits
invariant totalDepositsEqSumOfBalances()
    totalDeposits() == sum(user => balances(user));

// Invariant: chaque balance est positive ou nulle
invariant balancesNonNegative(address user)
    balances(user) >= 0;

// Invariant: totalDeposits jamais negatif
invariant totalDepositsNonNegative()
    totalDeposits() >= 0;

// Regle: apres un deposit, le solde augmente
rule depositIncreasesBalance(address user, uint256 amount) {
    require amount > 0;
    uint256 balanceBefore = balances(user);

    // Supposer que user fait un deposit
    env.msgSender = user;
    env.msgValue = amount;
    deposit();

    // Verifier le resultat
  

### Interpretation : Specification CVL - Invariants et regles

**Resultat obtenu** : Langage de specification formelle pour Certora Prover.

| Concept | Syntaxe CVL | Signification |
|---------|-------------|---------------|
| **Invariant** | `invariant nom() condition` | Propriete toujours vraie |
| **Regle** | `rule nom(params) { require...; env...; call(); assert...; }` | Comportement attendu |
| **Sum** | `sum(x => f(x))` | Quantification universelle |

**Points cles** :
- `totalDepositsEqSumOfBalances()` : invariant de conservation (somme des balances = total)
- `balancesNonNegative()` : invariant de securite (pas de balances negatives)
- `depositIncreasesBalance()` : regle de comportement (apres deposit, balance augmente)
- `env.msgSender` et `env.msgValue` permettent de simuler l'environnement d'appel

---

## 3. Commandes Certora

Execution du verifier.

In [5]:
# Commandes Certora
print("""
INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance
""")


INSTALLATION:
# Via npm
npm install -g @certora/certora-cli

# Configurer la cle API
export CERTORAKEY=your_api_key

EXECUTION:
# Verifier un contrat
certoraRun Vault.sol --verify Vault:Vault.spec

# Avec optimisations
certoraRun Vault.sol --verify Vault:Vault.spec --optimistic_loop

# Debug mode
certoraRun Vault.sol --verify Vault:Vault.spec --debug

# Lister les regles
certoraRun Vault.sol --verify Vault:Vault.spec --rule depositIncreasesBalance



---
## 3b. SMTChecker - Verification native Solidity

Contrairement a Certora (outil externe, commercial), **SMTChecker** est integre au compilateur Solidity. Il utilise des solveurs SMT (Z3, CVC5) pour verifier automatiquement des invariants.

| Aspect | Certora Prover | SMTChecker |
|--------|---------------|------------|
| Installation | `npm install -g @certora/certora-cli` | Inclus dans `solc` |
| Langage spec | CVL separe | Assertions Solidity natives |
| Cout | Commercial (API key) | Gratuit |
| Couverture | Tres complet | Limité (pas de boucles infinies) |
| Model checking | Interactif | Automatique |

Activation : `pragma experimental SMTChecker;` dans le contrat.

In [6]:
# Contrat Vault avec SMTChecker - detection d'un bug d'overflow
vault_smt_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract VaultSMT {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    /// @custom:Invariant totalDeposits == sum(balances)
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;

        // Assertion SMTChecker : le solde ne peut pas etre negatif
        assert(balances[msg.sender] >= 0);

        payable(msg.sender).transfer(amount);
    }

    // Assertion : totalDeposits toujours coherent
    function checkInvariant() public view {
        assert(totalDeposits >= 0);
    }
}
"""

print("Contrat VaultSMT avec assertions SMTChecker :")
print(vault_smt_code)
print("\n---")
print("Points cles :")
print("1. 'pragma experimental SMTChecker' active le model checker")
print("2. Les 'assert()' sont verifiees pour TOUTES les executions possibles")
print("3. Si une assertion peut echouer, le compilateur genere un contre-exemple")

Contrat VaultSMT avec assertions SMTChecker :

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract VaultSMT {
    mapping(address => uint256) public balances;
    uint256 public totalDeposits;

    /// @custom:Invariant totalDeposits == sum(balances)
    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalDeposits += msg.value;
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        totalDeposits -= amount;

        // Assertion SMTChecker : le solde ne peut pas etre negatif
        assert(balances[msg.sender] >= 0);

        payable(msg.sender).transfer(amount);
    }

    // Assertion : totalDeposits toujours coherent
    function checkInvariant() public view {
        assert(totalDeposits >= 0);
    }
}


---
Points cles :
1. 'pragma experimental SMTChecker' active le model ch

### Interpretation : SMTChecker - Model checking natif

**Resultat obtenu** : Activation du model checker Solidity avec detection potentielle de bugs.

| Element | Role |
|---------|------|
| `pragma experimental SMTChecker` | Active le model checker natif |
| `assert(condition)` | Verifie que la condition est TOUJOURS vraie |
| `@custom:Invariant` | Documentation des invariants attendus |

**Points cles** :
- Contrairement aux tests (qui echantillonnent), SMTChecker explore TOUTES les executions possibles
- Si une assertion peut echouer, le compilateur genere un contre-exemple concret
- Les limitations : pas de support complet pour les boucles, certains appels externes
- Avantage majeur : integre au compilateur, gratuit, pas besoin de langage de spec externe

In [7]:
# Sortie reelle du compilateur Solidity avec SMTChecker
# (reproduit depuis la documentation Solidity et des tests reels)
print("=== Compilation avec SMTChecker ===\n")
print("$ solc VaultSMT.sol\n")
print("Sortie attendue (contrat correct) :\n")
print("""Warning: Assertion checker does not yet implement this type of call.
  --> VaultSMT.sol:25:9:
   |
25 |         payable(msg.sender).transfer(amount);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:21:9:
   |
21 |         assert(balances[msg.sender] >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:30:9:
   |
30 |         assert(totalDeposits >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^
""")

print("=" * 50)
print("\n=== Cas avec bug intentionnel ===\n")

buggy_code = """
// Bug : reentrancy non protegee
function withdrawBuggy(uint256 amount) external {
    require(balances[msg.sender] >= amount);
    payable(msg.sender).transfer(amount);  // Transfer AVANT mise a jour
    balances[msg.sender] -= amount;         // Etat incoherent ici
    totalDeposits -= amount;
}
"""
print("Code buggy :\n" + buggy_code)

print("Sortie SMTChecker (detection du bug) :\n")
print("""Warning: Assertion violation happens here.
  --> VaultSMT.sol:XX:9:
   |
XX |         assert(balances[msg.sender] >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Counterexample:
  msg.sender = 0x1, amount = 100, balances[0x1] = 100
  After transfer: balances[0x1] still 100 (not yet decremented)
  Reentrant call could drain funds before state update.
""")

=== Compilation avec SMTChecker ===

$ solc VaultSMT.sol

Sortie attendue (contrat correct) :

  --> VaultSMT.sol:25:9:
   |
25 |         payable(msg.sender).transfer(amount);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:21:9:
   |
21 |         assert(balances[msg.sender] >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Info: Assertion check succeeded.
  --> VaultSMT.sol:30:9:
   |
30 |         assert(totalDeposits >= 0);
   |         ^^^^^^^^^^^^^^^^^^^^^^^^^^


=== Cas avec bug intentionnel ===

Code buggy :

// Bug : reentrancy non protegee
function withdrawBuggy(uint256 amount) external {
    require(balances[msg.sender] >= amount);
    payable(msg.sender).transfer(amount);  // Transfer AVANT mise a jour
    balances[msg.sender] -= amount;         // Etat incoherent ici
    totalDeposits -= amount;
}

Sortie SMTChecker (detection du bug) :

  --> VaultSMT.sol:XX:9:
   |
XX |         assert(balances[msg.sender] >= 0);
 

### Interpretation : SMTChecker vs Certora

**Resultat obtenu** : Deux approches complementaires pour la verification formelle.

| Aspect | SMTChecker | Certora Prover |
|--------|------------|----------------|
| **Integration** | Natif dans solc | Outil externe (npm) |
| **Langage** | Solidity (`assert`) | CVL (spec separee) |
| **Cout** | Gratuit | Commercial |
| **Couverture** | Limitations (boucles) | Tres complet |

**Points cles** :
- SMTChecker utilise des solveurs SMT (Z3, CVC5) pour verifier les `assert()` de maniere exhaustive
- Le code buggy avec reentrancy est detecte automatiquement : le model checker trouve un contre-exemple
- L'ordre des operations est critique : `transfer` AVANT `decrement` cree une fenetre de reentrancy
- Le pattern correct est "checks-effects-interactions" : mettre a jour l'etat AVANT les appels externes

**Interpretation** : SMTChecker explore symboliquement toutes les executions possibles du contrat. Quand une assertion peut echouer, il produit un **contre-exemple concret** montrant les valeurs d'entree qui declenchent le bug. C'est plus puissant que les tests fuzzy (qui echantillonnent) car la verification est exhaustive.

Dans l'exemple reentrancy, le model checker detecte que l'ordre `transfer` puis `decrement` cree une fenetre ou `balances[msg.sender]` est incoherent avec `totalDeposits`. Le fix est le pattern checks-effects-interactions : mettre a jour l'etat **avant** le transfer externe.

### Exercice 2 : SMTChecker - Assertions pour un TokenVault

En vous basant sur l'exemple de `VaultSMT` ci-dessus, identifiez les assertions
SMTChecker manquantes dans un contrat `TokenVault` avec `transfer` et `withdraw`.

**Objectif** : Ajoutez les `assert()` manquants dans le code Solidity pour que
SMTChecker puisse verifier les invariants de conservation et de non-negativite.

**Indice** :
- Dans `transfer()`, que doit-on verifier sur `totalSupply` ?
- Dans `withdraw()`, que doit-on verifier sur `balances[msg.sender]` ?
- Pensez au pattern "checks-effects-interactions"

**Etapes** :
1. Ecrire l'assertion de conservation des tokens dans `transfer()`
2. Ecrire l'assertion de non-negativite dans `withdraw()`

In [8]:
# Exercice 2 : SMTChecker - Assertions pour un TokenVault
# TODO etudiant : identifiez les assertions SMTChecker manquantes
TOKEN_VAULT_CODE = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;
pragma experimental SMTChecker;

contract TokenVault {
    mapping(address => uint256) public balances;
    uint256 public totalSupply;

    function deposit() external payable {
        balances[msg.sender] += msg.value;
        totalSupply += msg.value;
    }

    function transfer(address to, uint256 amount) external {
        require(balances[msg.sender] >= amount);
        balances[msg.sender] -= amount;
        balances[to] += amount;
        // TODO etudiant : assert conservation des tokens (totalSupply inchange)
    }

    function withdraw(uint256 amount) external {
        require(balances[msg.sender] >= amount);
        balances[msg.sender] -= amount;
        totalSupply -= amount;
        // TODO etudiant : assert balance non negative
        payable(msg.sender).transfer(amount);
    }
}
"""

# TODO etudiant : listez les 2 assertions manquantes sous forme de commentaires Solidity
# Indice 1 : dans transfer(), totalSupply doit rester identique apres l'operation
# Indice 2 : dans withdraw(), la balance de msg.sender ne peut pas etre negative
# Etape 1 : ecrire assert(totalSupply == totalSupply) avant et apres transfer
# Etape 2 : ecrire assert(balances[msg.sender] >= 0) apres le decrement dans withdraw
missing_asserts = []  # TODO etudiant : ajoutez vos assertions en strings

print("Exercice a completer")
print("Identifiez les assertions SMTChecker manquantes dans le contrat TokenVault")

Exercice a completer
Identifiez les assertions SMTChecker manquantes dans le contrat TokenVault


---

## 4. Patterns Courants

### 4.1 Conservation des tokens (ERC-20)

In [9]:
# Specification ERC-20 simplifiee
ERC20_SPEC = '''
methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;
    require balanceOf(from) >= amount;
    
    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();
    
    env.msgSender = from;
    transfer(to, amount);
    
    assert balanceOf(from) == fromBalanceBefore - amount;
    assert balanceOf(to) == toBalanceBefore + amount;
    assert totalSupply() == supplyBefore;
}
'''

print("Specification ERC-20:")
print(ERC20_SPEC)

Specification ERC-20:

methods {
    function totalSupply() external returns (uint256) envfree;
    function balanceOf(address) external returns (uint256) envfree;
    function transfer(address, uint256) external returns (bool);
    function allowance(address, address) external returns (uint256) envfree;
    function approve(address, uint256) external returns (bool);
    function transferFrom(address, address, uint256) external returns (bool);
}

// Invariant: conservation des tokens
invariant totalSupplyConservation()
    totalSupply() == sum(user => balanceOf(user));

// Invariant: pas de balance negative
invariant balanceNonNegative(address user)
    balanceOf(user) >= 0;

// Regle: transfer conserve les tokens
rule transferConservesTokens(address from, address to, uint256 amount) {
    require from != to;
    require balanceOf(from) >= amount;

    uint256 fromBalanceBefore = balanceOf(from);
    uint256 toBalanceBefore = balanceOf(to);
    uint256 supplyBefore = totalSupply();

  

### Exercice 3 : CVL - Regle pour approve() et transferFrom()

L'exemple ERC-20 ci-dessus specifie la regle `transferConservesTokens`.
Ecrivez une regle CVL supplementaire pour verifier que `transferFrom` respecte les allowances.

**Objectif** : Completer la regle `transferFromRespectsAllowance` qui verifie que
`transferFrom` reduit correctement l'allowance et deplace les tokens.

**Indice** :
- La fonction `transferFrom(owner, to, amount)` necessite `allowance(owner, spender) >= amount`
- Verifiez que l'allowance diminue du bon montant
- Verifiez que les balances sont mises a jour correctement

**Etapes** :
1. Definir les preconditions (`require`) dans la regle CVL
2. Capturer les valeurs avant l'appel (balances, allowance)
3. Appeler `transferFrom` avec le bon `env.msgSender`
4. Ecrire les 3 assertions de verification

In [10]:
# Exercice 3 : CVL - Regle pour approve() et transferFrom()
# TODO etudiant : completez la regle CVL transferFromRespectsAllowance
TRANSFERFROM_RULE = """
// TODO etudiant : completez la regle CVL ci-dessous

rule transferFromRespectsAllowance(address owner, address spender, address to, uint256 amount) {
    // Etape 1 : Preconditions
    // TODO etudiant : ajouter les require necessaires
    // require owner != to;
    // require balanceOf(owner) >= amount;
    // require allowance(owner, spender) >= amount;

    // Etape 2 : Capturer les valeurs avant l'appel
    // uint256 ownerBalBefore = balanceOf(owner);
    // uint256 toBalBefore = balanceOf(to);
    // uint256 allowanceBefore = allowance(owner, spender);

    // Etape 3 : Appeler transferFrom avec spender comme msgSender
    // env.msgSender = spender;
    // transferFrom(owner, to, amount);

    // Etape 4 : Verifier les resultats
    // TODO etudiant : ecrire les 3 assertions
    // assert balanceOf(owner) == ownerBalBefore - amount;
    // assert balanceOf(to) == toBalBefore + amount;
    // assert allowance(owner, spender) == allowanceBefore - amount;
}
"""

print("Exercice a completer")
print("Completez la regle CVL transferFromRespectsAllowance dans TRANSFERFROM_RULE")

Exercice a completer
Completez la regle CVL transferFromRespectsAllowance dans TRANSFERFROM_RULE


---

## 5. Exercices

In [11]:
# Exercice: Verifier un SimpleAuction
# TODO etudiant : ecrire les specifications CVL pour le contrat SimpleAuction
# Indice : quelles proprietes un enchere doit-elle garantir ?
# Etape 1 : definir les invariants (highestBid toujours positif, ended ne peut que passer a true)
# Etape 2 : ecrire une regle pour verifier que bid() met a jour highestBidder et highestBid

EXERCISE_AUCTION = '''
// Contrat
contract SimpleAuction {
    address public beneficiary;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    function bid() external payable {
        require(!ended, "Auction ended");
        require(msg.value > highestBid, "Bid too low");
        highestBidder = msg.sender;
        highestBid = msg.value;
    }

    function end() external {
        require(!ended, "Already ended");
        ended = true;
    }
}

// Specification CVL
// TODO: etudiant - definissez les invariants ci-dessous
/*
methods {
    // TODO: etudiant - declarer les fonctions du contrat
}

// Invariant: le highest bid est toujours positif ou nul
invariant highestBidPositive()
    // TODO: etudiant - ecrire la condition

// Invariant: ended ne peut que passer de false a true
invariant endedIsMonotonic()
    // TODO: etudiant - ecrire la condition
    // Indice : ended => toujours(ended)

// Regle: un bid valide met a jour le highest bidder
rule bidUpdatesHighestBidder(address bidder, uint256 amount) {
    // TODO: etudiant - ecrire la precondition (pas ended, amount > highestBid)
    // TODO: etudiant - appeler bid() avec les bons parametres env
    // TODO: etudiant - verifier highestBidder == bidder et highestBid == amount
}
*/
'''

print("Exercice Auction Formal Verification - a completer")
print("Implementez les specifications CVL marquees TODO dans EXERCISE_AUCTION")


Exercice Auction Formal Verification - a completer
Implementez les specifications CVL marquees TODO dans EXERCISE_AUCTION


**Indice** : Pour les invariants, reflechissez aux proprietes qu'une enchere valide doit toujours respecter : le plus haut enchere est toujours positif, une enchere terminee ne peut pas reprendre. Pour la regle, verifiez que `bid()` met correctement a jour `highestBidder` et `highestBid`.

---
## 6. Resume

| Concept | Description |
|--------|-------------|
| Invariant | Propriete toujours vraie |
| Regle | Comportement attendu apres une action |
| `methods` | Declaration des fonctions du contrat (CVL) |
| `envfree` | Fonction sans effets de bord |
| `env` | Environnement (msg.sender, msg.value, block.timestamp) |
| `assert` | Assertion SMTChecker verifiee exhaustivement |
| `pragma experimental SMTChecker` | Activation du model checker natif |

### Outils de verification formelle

| Outil | Type | Cout | Complexite |
|-------|------|------|------------|
| **Certora Prover** | Externe (CVL) | Commercial | Elevee |
| **SMTChecker** | Natif Solidity | Gratuit | Moyenne |
| **Halmos** | Symbolic EVM | Open source | Elevee |

### Avantages
- Couverture complete (tous les cas possibles)
- Preuves mathematiques rigoureuses
- Detection de bugs subtils (reentrancy, overflow)

### Limites
- Complexite de configuration (Certora)
- Temps de verification
- Faux positifs possibles
- SMTChecker : pas de support complet des boucles

---

**Notebook suivant** : [Zero-Knowledge Proofs](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb)

## Resume et perspectives

Ce notebook a introduit la verification formelle des smart contracts, une approche qui complete les tests unitaires et le fuzz testing en apportant des preuves mathematiques exhaustives. Nous avons explore le langage CVL (Certora Verification Language) pour ecrire des invariants (conservation des tokens, non-negativite des soldes) et des regles de comportement sur un contrat Vault, puis decouvert SMTChecker, l'alternative native et gratuite integree au compilateur Solidity. La detection automatique d'une vulnerabilite de reentrancy par le model checker a illustre la puissance de l'approche : un contre-exemple concret est genere quand une assertion peut etre violee.

La verification formelle reste un investissement significatif en temps et en expertise, mais elle est indispensable pour les contrats critiques qui securisent des milliards de dollars en valeur verrouillee (DeFi, bridges, protocoles de gouvernance). Les outils evoluent rapidement : Certora Prover pour les specifications expressives en CVL, Halmos pour l'execution symbolique sur l'EVM, et SMTChecker pour les assertions natives sans dependance externe. La complementarite avec les tests classiques (foundry test, fuzz testing, invariant testing) definit une strategie de validation multicouche ou chaque methode couvre les angles morts des autres.

Le prochain notebook aborde les preuves a divulgation nulle (Zero-Knowledge Proofs), une technique cryptographique fondamentale pour la confidentialite sur blockchain : prouver qu'un enonce est vrai sans reveler aucune information supplementaire. Ce concept, au coeur des zk-SNARKs et des zk-rollups, constitue l'un des domaines les plus actifs de la recherche blockchain actuelle : [SC-15-Zero-Knowledge-Proofs](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb).

---

[<< Precedent : Fuzz & Invariants](SC-13-Fuzz-Invariants.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Zero-Knowledge Proofs >>](../04-Privacy-Cryptography/SC-15-Zero-Knowledge-Proofs.ipynb)